# Module 2: Snowflake Postgres — Customer Self-Service Portal

This module extends the EPOWER demo with a **Snowflake Postgres** instance that backs the **"Mein EPOWER"** customer self-service portal — a web application where 20,000 customers manage their energy accounts online.

---

### The Business Case

Every energy retailer needs a digital customer portal. In Germany, where EPOWER operates, regulatory requirements (Marktkommunikation) and customer expectations demand self-service capabilities: meter reading submission, tariff switching, billing inquiries, and program enrollments.

**"Mein EPOWER"** is EPOWER's customer-facing web application. Behind it: a PostgreSQL database handling the transactional workload — logins, form submissions, order processing, and session management for 20,000 customers.

The challenge: **How do you get operational portal data into your analytical platform without building ETL pipelines?** The answer: Snowflake Postgres + pg_lake. The portal writes to Postgres for operations; Postgres writes to Iceberg for analytics; Snowflake reads Iceberg natively. Zero middleware.

---

### Why Postgres?

The portal is a standard web application stack: **React frontend → REST API → PostgreSQL**. This is the most common backend pattern in software — used by millions of applications worldwide.

| Portal Feature | OLTP Requirement | Why Not Snowflake? |
|---------------|-----------------|-------------------|
| **Submit meter readings** | Validation (new ≥ previous), INSERT with constraints | Sub-second response needed for UX |
| **Request tariff switch** | Atomic order with status lifecycle (PENDING → CONFIRMED → ACTIVE) | Row-level locking for concurrent status updates |
| **Open service request** | INSERT with auto-categorization | Hundreds of concurrent form submissions |
| **VPP enrollment** | Multi-step signup with rollback on failure | Transaction semantics (all-or-nothing) |
| **Login & sessions** | Concurrent auth, session UPDATEs, CSRF tokens | Millisecond reads for session validation |

---

### The Operational → Analytical Bridge

| Data | Lives in Postgres | Flows to Snowflake? | Why? |
|------|------------------|--------------------| -----|
| User sessions & auth | ✅ | ❌ | Ephemeral, no analytical value |
| Meter readings (raw) | ✅ | ❌ | Already in Snowflake via billing pipeline |
| Tariff orders (mutable) | ✅ | ❌ | Status changes frequently, needs row-locking |
| **Portal activity log** | ✅ | **✅ via pg_lake** | Append-only, denormalized, analytically rich |

The `portal_activity_log` is the bridge — every user action generates one immutable log entry. This append-only stream is the natural replication target for analytics.

---

### Architecture

```
+-------------------------------------------------------------+
| "Mein EPOWER" Portal (Web Application)                      |
| React Frontend -> REST API -> Snowflake Postgres            |
|                                                             |
| +--------------+ +----------------+ +--------------------+  |
| | portal_users | | meter_readings | | tariff_orders      |  |
| | (sessions)   | | (kWh data)     | | service_requests   |  |
| +--------------+ +----------------+ +--------------------+  |
|                                                             |
| Every user action -> portal_activity_log (append-only)      |
|                             |                               |
|                  pg_incremental (1 min)                     |
|                             v                               |
|                  +--------------------+                     |
|                  | Iceberg table      |                     |
|                  | (pg_lake managed)  |                     |
|                  +---------+----------+                     |
+----------------------------+--------------------------------+
                             |
                  Catalog Integration (auto-refresh 30s)
                             v
+-------------------------------------------------------------+
| SNOWFLAKE (Analytics + AI)                                  |
| EPOWER_BRONZE -> PORTAL_ACTIVITY_LOG (Iceberg)              |
| EPOWER_GOLD   -> MART_PORTAL_ENGAGEMENT                     |
|               -> PORTAL_SEMANTIC_VIEW -> EPOWER AGENT       |
+-------------------------------------------------------------+
```

---

### Snowflake Features Introduced

| Feature | What It Is | Role in This Module |
|---------|-----------|-------------------|
| **Snowflake Postgres** | Fully managed PostgreSQL. Connects via `psql`, JDBC, or any ORM. | Portal transactional backend |
| **pg_lake** | Postgres extension for native Iceberg table support | Creates Iceberg table for the activity log |
| **pg_incremental** | Automated, exactly-once incremental pipelines | Syncs activity log heap → Iceberg every minute |
| **Catalog Integration** | Snowflake reads external Iceberg catalogs | Bridge from Postgres-managed Iceberg to Snowflake |
| **Auto-Refresh** | Snowflake polls Iceberg catalog for new snapshots | Near real-time portal data in Snowflake |

---

### What You'll Learn

| Section | What we do |
|---------|------------|
| **§1** Prerequisites | Verify Module 1 is deployed |
| **§2** Postgres Instance | Provision Snowflake Postgres |
| **§3** psql Setup | Install psql, save connection |
| **§4** Portal Schema + Data | Create tables and seed data (in psql) |
| **§5** Catalog Integration | Connect Snowflake to Postgres Iceberg |
| **§6** Analytics Model | Create engagement metrics |
| **§7** Semantic View + Agent | Add portal_analyst tool |
| **§8** Verification & Demo | Validate pipeline, demo live sync |

**Runtime**: ~15 minutes | **Prerequisite**: Module 1 (`epower_hol.ipynb`) must be deployed

## 1. Prerequisites

This module requires Module 1 (`epower_hol.ipynb`) to be fully deployed — we need the customer dimension and product data.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected as: {session.get_current_user()}")

In [ ]:
%%sql
USE ROLE EPOWER_ROLE;
USE WAREHOUSE EPOWER_COMPUTE;
USE DATABASE EPOWER_DEMO;

SELECT 'CUSTOMER_DIM' AS required_object, COUNT(*) AS "ROWS" FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM
UNION ALL SELECT 'PRODUCT_DIM', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.PRODUCT_DIM
UNION ALL SELECT 'CUSTOMER_PRODUCTS', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_PRODUCTS
ORDER BY required_object;

## 2. Create Snowflake Postgres Instance

We provision a Snowflake Postgres instance to serve as the **portal backend** — the transactional database behind the "Mein EPOWER" web application.

&nbsp;

> **Snowflake Feature:** Snowflake Postgres is fully managed PostgreSQL. The portal's web application connects to it like any standard Postgres database — via `psql`, JDBC, or any ORM. No infrastructure to manage.

In [ ]:
%%sql
use role accountadmin;
CREATE POSTGRES INSTANCE IF NOT EXISTS MY_EPOWER_PORTAL
    COMPUTE_FAMILY = 'STANDARD_L'
    STORAGE_SIZE_GB = 10
    AUTHENTICATION_AUTHORITY = POSTGRES
    COMMENT = 'Mein EPOWER — customer self-service portal backend (20K customers)';

-- IMPORTANT: Save the credentials from the output! They cannot be retrieved later.

In [ ]:
%%sql
CREATE NETWORK RULE IF NOT EXISTS EPOWER_PG_INGRESS
    TYPE = IPV4
    VALUE_LIST = ('0.0.0.0/0')
    MODE = POSTGRES_INGRESS;

CREATE NETWORK POLICY IF NOT EXISTS EPOWER_PG_POLICY
    ALLOWED_NETWORK_RULE_LIST = ('EPOWER_PG_INGRESS');

ALTER POSTGRES INSTANCE MY_EPOWER_PORTAL
    SET NETWORK_POLICY = 'EPOWER_PG_POLICY';

DESCRIBE POSTGRES INSTANCE MY_EPOWER_PORTAL;

## 3. Connect to Postgres via psql

The Postgres schema, data loading, and pg_lake setup are done in a **standard PostgreSQL client**. This mirrors real-world usage: application developers work in their PG tools, data engineers work in Snowflake.

---

### Install psql on macOS

```bash
brew install libpq
brew link --force libpq
```

Verify: `psql --version` should show PostgreSQL 16+.

**Alternative GUI clients:** [DBeaver Community](https://dbeaver.io/) (free, cross-platform), [pgAdmin 4](https://www.pgadmin.org/).

---

### Save Connection (recommended)

To avoid typing the long connection string every time, save it in PostgreSQL's standard service file.

**1. Create/edit `~/.pg_service.conf`:**

```ini
[my_epower_portal]
host=<HOST from DESCRIBE output above>
port=5432
dbname=postgres
user=snowflake_admin
sslmode=require
```

**2. Save password in `~/.pgpass`:**

```
<HOST>:5432:postgres:snowflake_admin:<PASSWORD from CREATE output>
```

Then set permissions: `chmod 600 ~/.pgpass`

**3. Connect:**

```bash
psql service=my_epower_portal
```

---

### Quick Connect (one-liner)

If you prefer not to save the connection:

```bash
psql "host=<HOST> port=5432 dbname=postgres user=snowflake_admin sslmode=require"
```

You'll be prompted for the password.

## 4. Portal Schema + Data (Postgres Client)

This section is executed in your **Postgres client** (psql, DBeaver, pgAdmin), not in this notebook.

All SQL files are included in the `hol-module2/` folder of this repository.

### Step 1: Create the schema + pg_lake setup

```bash
psql service=my_epower_portal -f portal_postgres_setup.sql
```

This creates:
- 5 tables: `portal_users`, `meter_readings`, `tariff_orders`, `service_requests`, `portal_activity_log`
- Indexes for common query patterns
- pg_lake + pg_cron + pg_incremental extensions
- `portal_activity_log_iceberg` (Iceberg mirror table)

### Step 2: Load seed data

```bash
psql service=my_epower_portal -f portal_seed_data.sql
```

This inserts 20,000 portal users, 60 days of activity logs, meter readings, tariff orders, and service requests — all pre-generated from the EPOWER customer and product dimensions. Timestamps use `now() - interval` so dates are always relative to execution time.

### Step 3: Configure and start Iceberg sync

```bash
psql service=my_epower_portal -f portal_iceberg_sync.sql
```

This script:
1. **Bulk-copies** all existing `portal_activity_log` data into the Iceberg table (instant)
2. **Creates** a `pg_incremental` pipeline that syncs new rows every 1 minute going forward

After this step, heap and iceberg row counts should match immediately.

### Step 4: Verify in Postgres

```sql
SELECT
    (SELECT count(*) FROM portal_activity_log) AS heap_rows,
    (SELECT count(*) FROM portal_activity_log_iceberg) AS iceberg_rows;
```

Both counts should match. The pipeline is now active — any new INSERTs into `portal_activity_log` will appear in the Iceberg table within ~60 seconds.

---

### Pause: Complete the Postgres setup before continuing

Run the following in your **Postgres client** (psql, DBeaver, pgAdmin):

```bash
psql service=my_epower_portal -f portal_postgres_setup.sql
psql service=my_epower_portal -f portal_seed_data.sql
psql service=my_epower_portal -f portal_iceberg_sync.sql
```

The last script bulk-copies data to Iceberg and starts the sync pipeline. Verify immediately:

```sql
SELECT (SELECT count(*) FROM portal_activity_log) AS heap_rows,
       (SELECT count(*) FROM portal_activity_log_iceberg) AS iceberg_rows;
```

Once both counts match, continue with the next cell.

---

## 5. Snowflake Catalog Integration

Connect Snowflake to the Postgres-managed Iceberg table. After this, portal activity data is queryable in Snowflake — with auto-refresh every 30 seconds.

In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE CATALOG INTEGRATION PORTAL_POSTGRES_CATALOG
  CATALOG_SOURCE    = SNOWFLAKE_POSTGRES
  TABLE_FORMAT      = ICEBERG
  CATALOG_NAMESPACE = 'public'
  REST_CONFIG = (
    POSTGRES_INSTANCE      = 'MY_EPOWER_PORTAL'
    CATALOG_NAME           = 'postgres'
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
  )
  ENABLED = TRUE;

In [ ]:
GRANT USAGE ON INTEGRATION PORTAL_POSTGRES_CATALOG TO ROLE EPOWER_ROLE;

USE ROLE EPOWER_ROLE;
USE WAREHOUSE EPOWER_WH;

In [ ]:
%%sql
CREATE OR REPLACE ICEBERG TABLE EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG
    CATALOG = 'PORTAL_POSTGRES_CATALOG'
    CATALOG_TABLE_NAME = 'portal_activity_log_iceberg';

ALTER ICEBERG TABLE EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG
    SET AUTO_REFRESH = TRUE;

In [ ]:
%%sql
SELECT
    count(*) AS total_events,
    count(DISTINCT customer_key) AS unique_customers,
    min(event_time) AS earliest,
    max(event_time) AS latest,
    count(CASE WHEN event_type = 'LOGIN' THEN 1 END) AS logins,
    count(CASE WHEN event_type = 'METER_READING' THEN 1 END) AS readings,
    count(CASE WHEN event_type = 'TARIFF_CHANGE' THEN 1 END) AS tariff_changes,
    count(CASE WHEN event_type = 'SERVICE_REQUEST' THEN 1 END) AS service_requests
FROM EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG;

## 6. Analytics Model

We deploy a **separate dbt project** (`epower_dbt_portal/`) for the portal analytics — isolated from the Module 1 pipeline (`epower_dbt/`) so that Module 1 never depends on Postgres data. This Gold-layer mart aggregates portal activity into **daily engagement metrics** by region and customer type — answering questions like "How active is our portal?" and "Which regions have low digital adoption?"

### Data Pipeline

```
+-------------------------------------------------------------------------+
|  SNOWFLAKE POSTGRES (MY_EPOWER_PORTAL)                                  |
|                                                                         |
|  portal_activity_log --> pg_incremental (1 min) --> Iceberg table        |
|  (heap, OLTP)              $1/$2 time windows       (pg_lake managed)   |
+------------------------------------+------------------------------------+
                                     |
                    Catalog Integration (SNOWFLAKE_POSTGRES)
                    Auto-refresh: 30 seconds
                                     |
                                     v
+-------------------------------------------------------------------------+
|  SNOWFLAKE                                                              |
|                                                                         |
|  EPOWER_BRONZE.PORTAL_ACTIVITY_LOG  <-- Iceberg table (catalog-linked)  |
|        |                                                                |
|        |  dbt project: epower_dbt_portal                                |
|        |  (GROUP BY date, region, customer_type, event_type)            |
|        v                                                                |
|  EPOWER_GOLD.MART_PORTAL_ENGAGEMENT                                     |
|        |                                                                |
|        |  Columns: activity_date, region, customer_type,                |
|        |           event_type, event_count, unique_customers            |
|        v                                                                |
|  EPOWER_GOLD.PORTAL_SEMANTIC_VIEW  --> Cortex Analyst (text-to-SQL)     |
+-------------------------------------------------------------------------+
```

In [ ]:
# Deploy portal dbt project — auto-detect workspace name

user = session.sql("SELECT CURRENT_USER()").collect()[0][0]
db = f"USER${user}"
workspaces = session.sql(f"SHOW WORKSPACES IN SCHEMA {db}.PUBLIC").collect()
workspace_name = None

for ws in workspaces:
    name = ws["name"]
    try:
        check = session.sql(
            f"""LIST 'snow://workspace/{db}.PUBLIC."{name}"/versions/live/epower_dbt_portal/dbt_project.yml'"""
        ).collect()
        if len(check) > 0:
            workspace_name = name
            break
    except:
        continue

if workspace_name is None:
    raise RuntimeError(
        "Could not find a workspace containing 'epower_dbt_portal/'. "
        "Please ensure your workspace has the epower_dbt_portal folder at its root."
    )

workspace_uri = f'snow://workspace/{db}.PUBLIC."{workspace_name}"/versions/live/epower_dbt_portal'
print(f"Detected workspace: {workspace_name}")
print(f"Deploying from:     {workspace_uri}")

session.sql(f"""CREATE OR REPLACE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_PORTAL_PROJECT
    FROM '{workspace_uri}'""").collect()
print("✓ Portal dbt project deployed")

session.sql("EXECUTE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_PORTAL_PROJECT ARGS = 'run'").collect()
print("✓ Portal dbt project executed")

In [ ]:
%%sql
SELECT activity_date, region, event_type, event_count, unique_customers
FROM EPOWER_DEMO.EPOWER_GOLD.MART_PORTAL_ENGAGEMENT
ORDER BY activity_date DESC
LIMIT 20;

## 7. Semantic View + Agent Update

We create a **PORTAL_SEMANTIC_VIEW** and add a `portal_analyst` tool to the Intelligence Agent — enabling natural language questions about portal engagement.

### Agent Architecture (with Portal Extension)

```
+-------------------------------------------------------------------------+
|                    SNOWFLAKE INTELLIGENCE AGENT                          |
|           (13 tools: 8 Analyst + 4 Search + Chart)                      |
+-------------------------------------------------------------------------+
                                    |
                   +----------------+----------------+
                   v                                 v
+-------------------------------+     +----------------------------------+
|       CORTEX ANALYST          |     |        CORTEX SEARCH             |
|        (Text-to-SQL)          |     |           (RAG)                  |
|                               |     |                                  |
|  energy_sales_analyst         |     |  energy_docs_search              |
|  billing_analyst              |     |  product_docs_search             |
|  customer_energy_analyst      |     |  service_docs_search             |
|  service_analyst              |     |  service_logs_search             |
|  hr_analyst                   |     +----------------------------------+
|  market_prices_analyst        |
|  vpp_telemetry_analyst        |
|  portal_analyst <-- NEW       |
+-------------------------------+
                   |
                   v
+-------------------------------+     +----------------------------------+
|   PORTAL_SEMANTIC_VIEW        |     |  7 existing Semantic Views       |
|                               |     |  (Sales, Billing, Customer,      |
|  Facts:                       |     |   Service, HR, Market Prices,    |
|    event_count                |     |   VPP Telemetry)                 |
|    unique_customers           |     +----------------------------------+
|  Dimensions:                  |
|    activity_date              |
|    region                     |
|    customer_type              |
|    event_type                 |
|  Metrics:                     |
|    total_events (SUM)         |
|    total_unique_customers     |
+-------------------------------+
                   |
                   v
+-------------------------------+
|  MART_PORTAL_ENGAGEMENT       |
|  (from Postgres via pg_lake)  |
+-------------------------------+
```

In [ ]:
%%sql
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.PORTAL_SEMANTIC_VIEW
  TABLES (
    ENGAGEMENT AS EPOWER_DEMO.EPOWER_GOLD.MART_PORTAL_ENGAGEMENT
      PRIMARY KEY (ACTIVITY_DATE, REGION, CUSTOMER_TYPE, EVENT_TYPE)
  )
  FACTS (
    ENGAGEMENT.EVENT_COUNT AS EVENT_COUNT
      WITH SYNONYMS = ('events', 'activities', 'actions', 'Aktionen', 'Aktivitäten')
      COMMENT = 'Number of portal events',
    ENGAGEMENT.UNIQUE_CUSTOMERS AS UNIQUE_CUSTOMERS
      WITH SYNONYMS = ('active users', 'aktive Nutzer', 'unique users', 'DAU')
      COMMENT = 'Distinct customers who performed this event type'
  )
  DIMENSIONS (
    ENGAGEMENT.ACTIVITY_DATE AS ACTIVITY_DATE
      WITH SYNONYMS = ('date', 'day', 'Datum', 'Tag')
      COMMENT = 'Date of portal activity',
    ENGAGEMENT.REGION AS REGION
      WITH SYNONYMS = ('region', 'Gebiet', 'Bundesland')
      COMMENT = 'German geographic region (Nord, Süd, West, Ost)',
    ENGAGEMENT.CUSTOMER_TYPE AS CUSTOMER_TYPE
      WITH SYNONYMS = ('segment', 'Kundentyp', 'customer segment')
      COMMENT = 'Customer segment (Privatkunde, Kleingewerbe, Gewerbekunde)',
    ENGAGEMENT.EVENT_TYPE AS EVENT_TYPE
      WITH SYNONYMS = ('action type', 'Aktionstyp', 'event', 'Ereignis')
      COMMENT = 'Type of portal action: LOGIN, METER_READING, TARIFF_CHANGE, SERVICE_REQUEST'
  )
  METRICS (
    ENGAGEMENT.TOTAL_EVENTS AS SUM(ENGAGEMENT.EVENT_COUNT)
      WITH SYNONYMS = ('total events', 'Gesamtaktionen')
      COMMENT = 'Total number of portal events',
    ENGAGEMENT.TOTAL_UNIQUE_CUSTOMERS AS SUM(ENGAGEMENT.UNIQUE_CUSTOMERS)
      WITH SYNONYMS = ('total active users', 'Gesamtnutzer')
      COMMENT = 'Total distinct customers'
  )
  COMMENT = 'Customer portal engagement analytics — sourced from Snowflake Postgres via pg_lake';

In [ ]:
%%sql
CREATE OR REPLACE AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT
WITH PROFILE='{ "display_name": "EPOWER AGENT" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are a data analyst for EPOWER Energie Deutschland.
    CRITICAL LANGUAGE RULE: Always respond in the SAME language as the user's question. If the user asks in English, respond entirely in English. If the user asks in German, respond entirely in German. Never switch languages mid-response. The only exception is if the user explicitly requests a response in a specific language. Note: Even though the underlying data contains German terms, your explanatory text and insights MUST be in the user's language.
    DATA ACCESS: Energy sales, billing/consumption, service tickets, HR data, day-ahead electricity market prices, VPP IoT telemetry, customer portal engagement, and documents.
  orchestration: |
    TOOL SELECTION:
    - Document questions → energy_docs_search, product_docs_search, service_docs_search
    - Consumption + products → customer_energy_analyst
    - Sales/contracts → energy_sales_analyst
    - Billing → billing_analyst
    - Service tickets → service_analyst
    - HR data → hr_analyst
    - Electricity market prices, day-ahead → epulse_prices_analyst
    - VPP telemetry, solar yield, battery SOC, grid import/export → vpp_telemetry_analyst
    - Portal activity, digital engagement, meter readings, tariff changes, logins → portal_analyst
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: energy_sales_analyst, description: "Contracts, products, sales, revenue"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: billing_analyst, description: "Consumption, billing, payments"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: customer_energy_analyst, description: "Consumption by product ownership"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: service_analyst, description: "Service tickets, complaints"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: hr_analyst, description: "HR data, salaries"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: market_prices_analyst, description: "Day-ahead electricity market prices"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: vpp_telemetry_analyst, description: "VPP IoT telemetry: solar yield, battery SOC, grid import/export"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: portal_analyst, description: "Customer portal engagement: logins, meter readings, tariff changes, service requests, VPP enrollments. Data from Snowflake Postgres via pg_lake."}
  - tool_spec: {type: cortex_search, name: energy_docs_search, description: "Energy policies, terms"}
  - tool_spec: {type: cortex_search, name: product_docs_search, description: "Product documentation"}
  - tool_spec: {type: cortex_search, name: service_docs_search, description: "Service handbook"}
  - tool_spec: {type: cortex_search, name: service_logs_search, description: "Historical tickets"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  energy_sales_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  billing_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  customer_energy_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  service_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  hr_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  market_prices_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  vpp_telemetry_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  portal_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.PORTAL_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  energy_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS", max_results: 5}
  product_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS", max_results: 5}
  service_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS", max_results: 5}
  service_logs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS", max_results: 5}
$$;

## 8. Verification & Demo

### Demo Questions for the Agent

| # | Question | What it tests |
|---|----------|---------------|
| 1 | *"How many customers used the portal this week?"* | Basic engagement metric |
| 2 | *"Welche Region hat die höchste Portal-Nutzung?"* | Regional comparison (German) |
| 3 | *"Show me the trend of meter reading submissions over the last 30 days"* | Time-series + charting |
| 4 | *"What are the most popular tariff switches?"* | Tariff change analysis |
| 5 | *"Compare portal engagement between residential and business customers"* | Segment comparison |
| 6 | *"Which customers submit meter readings but have never changed their tariff?"* | **Cross-tool**: portal + sales |

&nbsp;

> **Presenter tip:** Question 6 combines portal data (from Postgres) with sales data (from Snowflake-native tables) — demonstrating the unified platform value.

### Live Demo: Real-Time Sync

Simulate a customer using the portal RIGHT NOW, then watch the data appear in Snowflake within 30–60 seconds.

**In your Postgres client (psql / DBeaver), run:**

```sql
INSERT INTO portal_activity_log (customer_key, event_time, event_type, event_detail, city, region, customer_type)
SELECT
    customer_key,
    now(),
    (ARRAY['LOGIN', 'METER_READING', 'TARIFF_CHANGE', 'SERVICE_REQUEST'])[1 + (random() * 3)::int],
    '{"source": "live_demo"}',
    'Hamburg',
    'Nord',
    'Privatkunde'
FROM portal_users
ORDER BY random()
LIMIT 50;
```

Wait ~60 seconds, then run the next cell to verify the data arrived in Snowflake.

In [ ]:
%%sql
SELECT
    MAX(event_time) AS most_recent_event,
    DATEDIFF('second', MAX(event_time), CURRENT_TIMESTAMP()) AS seconds_ago,
    COUNT(*) AS total_rows
FROM EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG;

---

## Summary

In this module you built a complete **web application backend → analytics** pipeline:

| What | How |
|------|-----|
| **Portal backend** | Snowflake Postgres — users, meter readings, tariff orders, service requests |
| **Zero-ETL replication** | pg_lake + pg_incremental → Iceberg (no middleware) |
| **Near real-time** | Auto-refresh every 30 seconds |
| **Analytics** | Gold-layer engagement metrics by region, segment, and action type |
| **AI-ready** | Semantic View + Cortex Agent — queryable in natural language |

**The key insight:** The portal's web application needs Postgres for what web apps always need — low-latency CRUD, session management, form validation, transactional consistency. Snowflake Postgres provides this as a managed service. The analytics-relevant activity stream flows to Snowflake automatically via open standards (Iceberg) — no ETL pipeline to build or maintain.

---

*EPOWER Module 2 — Snowflake Postgres + pg_lake — Powered by Snowflake*